In [ ]:
!pip install torch transformers datasets accelerate peft bitsandbytes
!pip install wandb tensorboard seqeval scikit-learn
!pip install sentencepiece protobuf
!!pip install jsonlines
!pip install -U unsloth trl

In [ ]:
import os
import yaml
import torch
from transformers import (
    TrainingArguments,
    Trainer,
)
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
DATASETS_DIR = "./datasets"
OUTPUTS_DIR = "./outputs"

os.makedirs(DATASETS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

In [ ]:
from datasets import load_dataset
import json
import random
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
DATASET_NAME = "ai4privacy/pii-masking-200k"
OUTPUT_DIR = Path("./pii_dataset_as_is")
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
TRAIN_RATIO = 0.7
DEV_RATIO = 0.2
TEST_RATIO = 0.1

random.seed(SEED)

# -----------------------------
# Load Dataset (ONLY train exists)
# -----------------------------
dataset = load_dataset(DATASET_NAME, split="train")

# -----------------------------
# Process examples (labels unchanged)
# -----------------------------
processed = []

for item in dataset:
    entities = []

    for span in item["privacy_mask"]:
        entities.append({
            "label": span["label"],  # unchanged
            "text": item["source_text"][span["start"]:span["end"]],
            # Uncomment if needed
            # "start": span["start"],
            # "end": span["end"],
        })

    if entities:
        processed.append({
            "text": item["source_text"],
            "entities": entities
        })

# -----------------------------
# Shuffle & Split
# -----------------------------
random.shuffle(processed)

n = len(processed)
train_end = int(n * TRAIN_RATIO)
dev_end = train_end + int(n * DEV_RATIO)

train_data = processed[:train_end]
dev_data   = processed[train_end:dev_end]
test_data  = processed[dev_end:]

# -----------------------------
# Save JSONL
# -----------------------------
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(train_data, OUTPUT_DIR / "train.jsonl")
save_jsonl(dev_data, OUTPUT_DIR / "dev.jsonl")
save_jsonl(test_data, OUTPUT_DIR / "test.jsonl")

# -----------------------------
# Stats
# -----------------------------
print("✅ Done!")
print(f"Train: {len(train_data)}")
print(f"Dev:   {len(dev_data)}")
print(f"Test:  {len(test_data)}")


In [ ]:
dataset = load_dataset(
    "json",
    data_files={
        "train": os.path.join(OUTPUT_DIR,"train.jsonl"),
        "validation": os.path.join(OUTPUT_DIR,"dev.jsonl")
    }
)

In [ ]:
def setup_model_and_tokenizer():
    """
    Load model, tokenizer, and apply quantization + LoRA config if specified.

    Args:
        use_4bit (bool, optional): Override whether to load in 4-bit mode.
        use_lora (bool, optional): Override whether to apply LoRA adapters.

    Returns:
        tuple: (model, tokenizer)
    """
    model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
    print(f"\nLoading model: {model_name}")

    # ------------------------------
    # Tokenizer setup
    # ------------------------------
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Determine quantization + LoRA usage
    load_in_4bit = True
    apply_lora = 'lora_r'

    # ------------------------------
    # Quantization setup (optional)
    # ------------------------------

    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )


    # ------------------------------
    # Model loading
    # ------------------------------
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_cfg,
        device_map="auto", #### device_map = "balanced",
        dtype=torch.bfloat16

    )

    # ------------------------------
    # LoRA setup (optional)
    # ------------------------------
    
    print("🔧 Applying LoRA configuration...")
    model = prepare_model_for_kbit_training(model)
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    # model.print_trainable_parameters()


    return model, tokenizer


In [ ]:
model, tokenizer = setup_model_and_tokenizer()

In [ ]:
task_instruction = """

You are a precise information extraction system.

Your task is to extract personally identifiable information (PII) entities from the given text.

Extract all the entities that you find in the text

Rules:
1. Return ONLY valid JSON. Do NOT include explanations or extra text.
5. The "text" field MUST match the substring from start to end.
6. If no entities are found, return: {"entities": []}
7. Do NOT guess or infer missing information.

Output format:
{
  "entities": [
    {
      "entity_type": "<one of the allowed types>",
      "text": "<exact substring>"
    }
  ]
}

Text:
{text}

"""

In [ ]:
from datasets import load_dataset

def build_messages_for_sample(text, entities, task_instruction, include_assistant):
    
    messages = [
        {"role": "user", "content": task_instruction + " " + text},
        
    ]
    if include_assistant:
        messages.append({
            "role": "assistant",
            "content": json.dumps({"entities": entities}, ensure_ascii=False)
        })

    return messages

In [ ]:
def preprocess_samples(examples, tokenizer, task_instruction, max_length):
    """Tokenise dialogues and apply assistant-only masking for causal LM."""
    input_ids_list, labels_list, attn_masks = [], [], []

    for text, entities in zip(examples["text"], examples["entities"]):
        sample = {"text": text, "entities": entities}

        # Build chat-style text

        msgs_full = build_messages_for_sample(
            text, entities, task_instruction, include_assistant=True
        )
        msgs_prompt = build_messages_for_sample(
            text, entities, task_instruction, include_assistant=False
        )

        text_full = tokenizer.apply_chat_template(
            msgs_full, tokenize=False, add_generation_prompt=False
        )
        text_prompt = tokenizer.apply_chat_template(
            msgs_prompt, tokenize=False, add_generation_prompt=True
        )
        prompt_len = len(text_prompt)

        tokens = tokenizer(
            text_full,
            max_length=max_length,
            truncation=True,
            padding=False,
            add_special_tokens=False,
            return_offsets_mapping=True,
        )

        # Mask non-assistant tokens
        start_idx = len(tokens["input_ids"])
        for i, (start, _) in enumerate(tokens["offset_mapping"]):
            if start >= prompt_len:
                start_idx = i
                break

        labels = [-100] * start_idx + tokens["input_ids"][start_idx:]
        input_ids_list.append(tokens["input_ids"])
        labels_list.append(labels)
        attn_masks.append(tokens["attention_mask"])

    return {
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attn_masks,
    }

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class PaddingCollator:
    def __init__(self, tokenizer, label_pad_token_id=-100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, batch):
        # Convert lists to tensors
        input_ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in batch]
        attn_masks = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in batch]
        labels = [torch.tensor(f["labels"], dtype=torch.long) for f in batch]

        # Pad to the max length in this batch
        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        attn_masks = pad_sequence(attn_masks, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=self.label_pad_token_id)

        return {
            "input_ids": input_ids,
            "attention_mask": attn_masks,
            "labels": labels,
        }

In [ ]:
def tokenize_dataset(tokenizer, train_data, val_data):

    print("\nTokenizing datasets...")
    tokenized_train = train_data.map(
        lambda e: preprocess_samples(
            e, tokenizer, task_instruction, 512
        ),
        batched=True,
        remove_columns=train_data.column_names,
    )
    tokenized_val = val_data.map(
        lambda e: preprocess_samples(
            e, tokenizer, task_instruction, 512
        ),
        batched=True,
        remove_columns=val_data.column_names,
    )

    return tokenized_train, tokenized_val

In [ ]:
tokenized_train, tokenized_val = tokenize_dataset(tokenizer, dataset["train"], dataset["validation"])

In [ ]:
!pip install wandb weave

In [ ]:
import os
os.environ['WANDB_API_KEY'] = ''

In [ ]:
collator = PaddingCollator(tokenizer=tokenizer)

output_dir = os.path.join(OUTPUTS_DIR, "lora_samsum")
os.makedirs(output_dir, exist_ok=True)

args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    max_steps=500,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=float(2e-4),
    lr_scheduler_type="cosine",
    warmup_steps=100,
    bf16=True,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=25,
    save_total_limit=2,
    max_grad_norm=1.0,
    report_to="wandb",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

print("\nStarting LoRA fine-tuning...")
trainer.train()
print("\nTraining complete!")

save_dir = os.path.join(output_dir, "lora_adapters")
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Saved LoRA adapters to {save_dir}")

In [ ]:
from huggingface_hub import login

login(token="")

In [ ]:
def push_to_hub(
    model: PeftModel, tokenizer: AutoTokenizer, model_name: str, hf_username: str
):
    """
    Push a model and tokenizer to Hugging Face Hub.
    """
    model_id = f"{hf_username}/{model_name}"
    try:
        model.push_to_hub(f"{model_id}-adapters", private=False)

        merged_model = model.merge_and_unload()
        merged_model.push_to_hub(model_id, private=False)

        tokenizer.push_to_hub(model_id)
        print(f"Adapters successfully pushed to: https://huggingface.co/{model_id}")
    except Exception as e:
        print(f"Error pushing to Hugging Face: {e}")
        print("Make sure you're logged in with: huggingface-cli login")

In [ ]:
push_to_hub(model, tokenizer, "qwen-ner", "gr8nishan")